# NCAA Men's March Madness Decision Tree

This notebook builds a decision tree classifier to predict NCAA men's tournament game outcomes from pre-tournament KenPom team features.

Requested split:

- Training: 1997-2016
- Validation: 2017-2025

KenPom archive note: KenPom public historical ratings begin at 2002. The notebook keeps the requested split, but rows are only used when both teams can be joined to KenPom features. Subscriber-only Four Factors and Miscellaneous tables are scraped when `KENPOM_EMAIL` and `KENPOM_PASSWORD` are set in the environment.

In [ ]:
from pathlib import Path
import json
import os
import sys

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, brier_score_loss, log_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier, export_text

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

from src.download_ncaa_data import main as download_ncaa_data
from src.kenpom_scraper import combine_cached_years, scrape_year, KenPomClient
from src.train_decision_tree import (
    load_tournament_games,
    load_kenpom_features,
    numeric_feature_columns,
    attach_features,
    build_examples,
    evaluate,
)

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
MODEL_PATH = PROJECT_ROOT / 'models' / 'decision_tree_march_madness.joblib'
PREDICTIONS_PATH = PROJECT_ROOT / 'reports' / 'validation_predictions.csv'
METRICS_PATH = PROJECT_ROOT / 'reports' / 'metrics.json'

PROJECT_ROOT

## 1. Download NCAA Tournament Results

The model expects Kaggle-format men's tournament files: `MTeams.csv` and `MNCAATourneyCompactResults.csv`.

In [ ]:
needed_ncaa_files = [RAW_DIR / 'MTeams.csv', RAW_DIR / 'MNCAATourneyCompactResults.csv']
if not all(path.exists() for path in needed_ncaa_files):
    download_ncaa_data()
else:
    print('NCAA files already exist:', [path.name for path in needed_ncaa_files])

## 2. Scrape Or Combine KenPom Features

Set `RUN_KENPOM_SCRAPE = True` to scrape missing seasons from KenPom. If `KENPOM_EMAIL` and `KENPOM_PASSWORD` are available in the environment, the scraper will also attempt subscriber Four Factors and Miscellaneous pages. Without credentials, it uses public efficiency ratings only.

Recommended credential handling from a terminal before launching Jupyter:

```bash
export KENPOM_EMAIL='your-email'
export KENPOM_PASSWORD='your-password'
jupyter notebook
```

In [ ]:
START_YEAR = 2002
END_YEAR = 2025
RUN_KENPOM_SCRAPE = False
SCRAPE_DELAY_SECONDS = 10.0

has_kenpom_login = bool(os.getenv('KENPOM_EMAIL') and os.getenv('KENPOM_PASSWORD'))
public_efficiency_only = not has_kenpom_login

if RUN_KENPOM_SCRAPE:
    client = KenPomClient(os.getenv('KENPOM_EMAIL'), os.getenv('KENPOM_PASSWORD'), delay=SCRAPE_DELAY_SECONDS)
    client.login()
    for year in range(START_YEAR, END_YEAR + 1):
        print(f'Scraping KenPom {year}')
        scrape_year(
            client,
            year,
            include_team_pages=has_kenpom_login,
            public_efficiency_only=public_efficiency_only,
            use_cache=True,
        )

kenpom = combine_cached_years(START_YEAR, END_YEAR)
print(kenpom.shape)
kenpom.head()

## 3. Build Matchup-Level Training Rows

Each tournament game becomes two examples: winner-vs-loser labeled `1`, and loser-vs-winner labeled `0`. For every numeric KenPom feature, the model gets both the signed feature difference and the absolute difference.

In [ ]:
games = load_tournament_games()
kenpom = load_kenpom_features()
feature_cols = numeric_feature_columns(kenpom)

games_with_features = attach_features(games, kenpom, feature_cols)
x, y, meta = build_examples(games_with_features, feature_cols)

train_mask = meta['season'].between(1997, 2016)
valid_mask = meta['season'].between(2017, 2025)

summary = {
    'feature_count': len(feature_cols),
    'model_column_count': x.shape[1],
    'all_examples': int(len(x)),
    'train_examples': int(train_mask.sum()),
    'validation_examples': int(valid_mask.sum()),
    'actual_train_years': sorted(meta.loc[train_mask, 'season'].unique().tolist()),
    'actual_validation_years': sorted(meta.loc[valid_mask, 'season'].unique().tolist()),
}
summary

## 4. Train Decision Tree Model

In [ ]:
if train_mask.sum() == 0 or valid_mask.sum() == 0:
    raise ValueError('No train or validation rows after joining tournament games to KenPom features.')

model = Pipeline(
    steps=[
        ('imputer', ColumnTransformer([('num', SimpleImputer(strategy='median'), x.columns)], remainder='drop')),
        (
            'tree',
            DecisionTreeClassifier(
                max_depth=4,
                min_samples_leaf=25,
                criterion='log_loss',
                random_state=42,
            ),
        ),
    ]
)

model.fit(x.loc[train_mask], y.loc[train_mask])
model

## 5. Evaluate On Validation Seasons

In [ ]:
train_metrics = evaluate(model, x.loc[train_mask], y.loc[train_mask])
valid_metrics = evaluate(model, x.loc[valid_mask], y.loc[valid_mask])

metrics = {
    'requested_train_years': '1997-2016',
    'requested_validation_years': '2017-2025',
    'actual_train_years': summary['actual_train_years'],
    'actual_validation_years': summary['actual_validation_years'],
    'rows': {
        'train': int(train_mask.sum()),
        'validation': int(valid_mask.sum()),
        'all_examples': int(len(x)),
    },
    'metrics': {'train': train_metrics, 'validation': valid_metrics},
}

pd.DataFrame(metrics['metrics']).T

## 6. Inspect The Tree And Feature Importance

In [ ]:
tree_text = export_text(model.named_steps['tree'], feature_names=list(x.columns))
print(tree_text)

In [ ]:
importance = pd.DataFrame(
    {
        'feature': x.columns,
        'importance': model.named_steps['tree'].feature_importances_,
    }
).sort_values('importance', ascending=False)

importance.query('importance > 0').head(20)

## 7. Run 2026 Tournament Predictions

In [ ]:
from src.predict_2026_tournament import main as predict_2026_tournament

MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump({'model': model, 'features': list(x.columns)}, MODEL_PATH)
predict_2026_tournament()

summary_2026 = json.loads((PROJECT_ROOT / 'reports' / '2026_tournament_summary.json').read_text())
predictions_2026 = pd.read_csv(PROJECT_ROOT / 'reports' / '2026_tournament_predictions.csv')

pd.DataFrame(summary_2026['by_round']).T

In [ ]:
predictions_2026[[
    'round',
    'winner',
    'loser',
    'winner_score',
    'loser_score',
    'pred_actual_winner_prob',
    'predicted_winner',
    'prediction_correct',
]].head(20)

In [ ]:
predictions_2026.loc[
    ~predictions_2026['prediction_correct'],
    ['round', 'winner', 'loser', 'winner_score', 'loser_score', 'pred_actual_winner_prob', 'predicted_winner']
]

## 8. Generated Model Visualizations

These cells create saved visual explanations for the decision tree: feature importance, round accuracy, confidence vs margin, and a bracket-style HTML report for the 2026 tournament.

In [ ]:
from IPython.display import HTML, IFrame, Image, display
from src.visualize_model import main as create_model_visualizations

create_model_visualizations()


### Feature Importance Chart

This shows which matchup-difference features the decision tree actually used most. Higher bars mean the feature reduced impurity more across tree splits.

In [ ]:
display(Image(filename=str(PROJECT_ROOT / 'reports' / 'feature_importance.png')))


### Other Model Diagnostics

Round accuracy shows where the model held up or struggled. The confidence chart compares how sure the model was with the actual game margin.

In [ ]:
display(Image(filename=str(PROJECT_ROOT / 'reports' / '2026_accuracy_by_round.png')))
display(Image(filename=str(PROJECT_ROOT / 'reports' / '2026_prediction_confidence.png')))


### 2026 Bracket Diagram With Model Picks

The HTML bracket has one card per played matchup. Blue cards are correct picks, red cards are misses. Each card includes the model pick, confidence, top matchup gaps, and the decision-tree path that explains the pick.

In [ ]:
display(IFrame(src=str(PROJECT_ROOT / 'reports' / '2026_bracket_predictions.html'), width='100%', height=900))


### Matchup-Level Explanations

Use this table when you want the plain-text reason for a pick without opening the bracket diagram.

In [ ]:
explanations_2026 = pd.read_csv(PROJECT_ROOT / 'reports' / '2026_prediction_explanations.csv')
explanations_2026[[
    'round',
    'matchup',
    'predicted_winner',
    'predicted_winner_probability',
    'prediction_correct',
    'top_matchup_drivers',
    'decision_tree_rules',
]].head(20)


## 9. Save Model, Predictions, And Metrics

In [ ]:
valid_meta = meta.loc[valid_mask].copy()
valid_meta['pred_team_a_win_prob'] = model.predict_proba(x.loc[valid_mask])[:, 1]
valid_meta['pred_team_a_win'] = (valid_meta['pred_team_a_win_prob'] >= 0.5).astype(int)

MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
PREDICTIONS_PATH.parent.mkdir(parents=True, exist_ok=True)
METRICS_PATH.parent.mkdir(parents=True, exist_ok=True)

metrics['tree'] = tree_text
joblib.dump({'model': model, 'features': list(x.columns)}, MODEL_PATH)
valid_meta.to_csv(PREDICTIONS_PATH, index=False)
METRICS_PATH.write_text(json.dumps(metrics, indent=2))

print(f'Wrote {MODEL_PATH}')
print(f'Wrote {PREDICTIONS_PATH}')
print(f'Wrote {METRICS_PATH}')
valid_meta.head()